In [ ]:
import numpy as np
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp
import sys
from pathlib import Path
# add Modeling/ to Python path
project_root = Path.cwd().parents[2]
sys.path.append(str(project_root))
from Modeling.models.beam_properties import PiezoBeamParams
from Modeling.models.plotting import animate_field_1d
import matplotlib.pyplot as plt
from Modeling.models.ROM import ROM

import numpy as np
from numpy import pi	
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from IPython.display import clear_output
from matplotlib import cm, colors
import pandas as pd
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [ ]:

Q = 300
params = PiezoBeamParams( 
    hp=0.55e-3, hs=0.51e-3, b=21e-3, w_p=21e-3,
    Q = 31
    )
params.zeta_p = 0.0151/100
params.zeta_q = 0.0392/100
print('L_b= ',params.L_b)
print("YI= ",params.YI)
print("m= ",params.m, "theta= ", f'{params.theta_mech:0.1e}', "Cp= ",f"{params.Cp[0]:0.13e} nF")
print("c11= ", f'{1/params.s11*1e-9:0.1f} GPa')
print("e31= ", f'{params.e31:0.1f}', "eps=", f'{-params.eps33*1e9:0.1f}')  # in C/m^2
N = 200
rom = ROM(params, N=N)
rom.homogenized_parameters(K_c=-1.6e10, R_c=1e3, K_i=1800)

In [ ]:
# plt.figure()    
plt.plot(rom.omega[:10]/2/np.pi, '.-')
plt.show()

In [ ]:
%matplotlib widget
Q = 300
params = PiezoBeamParams( 
    hp=0.55e-3, hs=0.51e-3, b=21e-3, w_p=21e-3, w_s=1e-3,
    Q = Q
    )
params.zeta_p = 0.0151/100
params.zeta_q = 0.0392/100
N = 200
rom = ROM(params, N=N)


#%%
x_eval = np.linspace(0, rom.p.L_b, 500)

K_i = 200; K_p = 1e-4; K_c = 0

f0 = 200; f1 = 600

j_exc = 299
freq_modal, vel_mag, disp_mag, veloc_frq = rom.frequency_response(j_exc=j_exc, K_c=K_c, K_p=K_p, K_i=K_i,
										w=np.linspace(f0, f1, 500)*2*np.pi, x_eval=x_eval)
# %matplotlib widget
plt.figure(figsize=(12, 4))
# plt.semilogy(frq_linear_exp, np.mean(frf_data_linear_exp[:,:], axis=1), 'r--', label=' Experiment')
# plt.semilogy(comsol_OC['freq'], comsol_OC['w']*2*pi*comsol_OC['freq'], 'g-', label='COMSOL ')
# plt.semilogy(comsol_OC['freq'], comsol_OC['w'], 'g-', label='COMSOL displacement FRF')
# plt.semilogy(frq_OC_exp, np.mean(frf_data_OC_exp[:,:], axis=1), 'k--', label=f'Open circuit Exp.')
# plt.semilogy(frq_SC_exp, np.mean(frf_data_SC_exp[:,:], axis=1), 'b--', label=f'Short circuit Exp.')
# plt.semilogy(freq_modal, vel_mag, '.-', label='Modal Reduced Order'   )
# plt.semilogy(freq, FRF, '.-', linewidth=1.5, label='Time Domain ROM')
plt.semilogy(freq_modal, vel_mag, '.-', label='Frequency Domain ROM'   )
# plt.semilogy(frq_OC_exp, np.mean(frf_data_OC_exp[:,:], axis=1), 'k--', label=f'Experiment')
# plt.semilogy(frq_linear, np.mean(frf_data_linear[:,:], axis=1), 'r--', label=' Exp.')

# plt.semilogy(freq_modal, disp_mag*freq_modal*2*np.pi, '-', label='Modal Reduced Order Displacement $j \omega$'   )
plt.legend()
# plt.xlim([1300, 3000])
# plt.xlim([f0, f1])
# plt.ylim([1e-5, 1e-3])
# plt.ylim([3e-5, 6e-4])
plt.xlabel("Frequency [Hz]")
plt.ylabel("AverageVelocity/Voltage FRF")
plt.grid(True)
plt.show()


In [16]:
R_c = 1e3; K_c = 3e5; K_i = 220; K_p = 1e-5
ref_scales = params.nondimensional_scales(K_i=K_i, K_c=K_c, R_c=R_c)
hom_params = params.homogenized_parameters(K_c=K_c, R_c=R_c, K_i=K_i)
print(ref_scales)
print(hom_params)
print('L= ', R_c / K_i)
print('Lc= ', R_c / K_c)
print(ref_scales)
print(hom_params)

theta_tilde = ref_scales['theta_tilde']


{'t0': 0.0003294852327205354, 'x0': 0.02225121869321126, 'lambda0': 0.0270801280154532, 'w0': 6.386547231595514e-05, 'theta_tilde': -0.33866683001493875}
{'m_bar': 0.20447700000000005, 'EI_bar': 0.46172965047990305, 'Cp_bar': 1.1373006708475797e-06, 'L_bar': 0.09545454545454547, 'Lc_bar': 7.000000000000001e-05, 'theta_bar': -0.0002454167454167455}
L=  4.545454545454546
Lc=  0.0033333333333333335
{'t0': 0.0003294852327205354, 'x0': 0.02225121869321126, 'lambda0': 0.0270801280154532, 'w0': 6.386547231595514e-05, 'theta_tilde': -0.33866683001493875}
{'m_bar': 0.20447700000000005, 'EI_bar': 0.46172965047990305, 'Cp_bar': 1.1373006708475797e-06, 'L_bar': 0.09545454545454547, 'Lc_bar': 7.000000000000001e-05, 'theta_bar': -0.0002454167454167455}


In [ ]:

def q_fun_ndim(Omega):
    q4 = (Omega**4 - Omega**2) / ((1 + theta_tilde**2)*Omega**2 - 1)
    return q4**0.25

def q_fun_dim(omega0):
    omega0_ndim = omega0 * ref_scales['t0']
    q0 = q_fun_ndim(omega0_ndim)
    return q0 / ref_scales['x0']

def P_vg_fun_dim(branch):
    if branch == 'acoustic':
        Omega = np.linspace(0, 1/( 1 + theta_tilde**2)**0.5*0.99, 5000)
    elif branch == 'optic':
        Omega = np.linspace(1, 3.0, 5000)
    q = q_fun_ndim(Omega) 
    dOmega_dq = np.gradient(Omega, q)
    d2Omega_dq2 = np.gradient(dOmega_dq, q)
    from scipy.interpolate import interp1d
    dOmega_dq_func = interp1d(Omega, dOmega_dq, kind='cubic', fill_value='extrapolate')
    d2Omega_dq2_func = interp1d(Omega, d2Omega_dq2, kind='cubic', fill_value='extrapolate')
    P = lambda omega0: 0.5 * d2Omega_dq2_func(omega0*ref_scales['t0']) * ref_scales['x0'] **2 / ref_scales['t0']
    v_g = lambda omega0: dOmega_dq_func(omega0*ref_scales['t0']) * ref_scales['x0']  / ref_scales['t0']
    return v_g, P

def P_vg_fun_ndim(branch):
    if branch == 'acoustic':
        Omega = np.linspace(0, 1/( 1 + theta_tilde**2)**0.5*0.99, 5000)
    elif branch == 'optic':
        Omega = np.linspace(1, 3.0, 5000)
    q = q_fun_ndim(Omega) 
    dOmega_dq = np.gradient(Omega, q)
    d2Omega_dq2 = np.gradient(dOmega_dq, q)
    from scipy.interpolate import interp1d
    dOmega_dq_func = interp1d(Omega, dOmega_dq, kind='cubic', fill_value='extrapolate')
    d2Omega_dq2_func = interp1d(Omega, d2Omega_dq2, kind='cubic', fill_value='extrapolate')
    P = lambda omega0: 0.5 * d2Omega_dq2_func(omega0) 
    v_g = lambda omega0: dOmega_dq_func(omega0) 
    return v_g, P


def eigen_vector_ndim(omega0):
    q0 = q_fun_ndim(omega0)
    C_w1 = 1j * theta_tilde * omega0* q0**2  / (-omega0**2 + q0**4)
    C_Lambda1 = 1
    return np.array((C_w1, C_Lambda1))

def F(DT, DX): 
    F_matrix = np.array([
        [DT**2 + DX**4, -theta_tilde*DT*DX**2],
        [theta_tilde*DT*DX**2, DT**2 + 1]
    ])
    return F_matrix

def F_DT(DT, DX): 
    F_matrix = np.array([
        [2*DT, -theta_tilde*DX**2],
        [theta_tilde*DX**2, 2*DT]
    ])
    return F_matrix

def Q_fun_dim(omega0):
    q0 = q_fun_dim(omega0 )
    omega0_ndim = omega0 * ref_scales['t0']
    eig_vec = eigen_vector_ndim(omega0_ndim)
    m_bar = hom_params['m_bar']
    Cp_bar = hom_params['Cp_bar']
    F_matrix = np.array([
        [2*m_bar*omega0, 1j*theta_tilde*q0**2],
        [-1j*theta_tilde*q0**2, 2*Cp_bar*omega0]
    ])
    coeff = - 3 / (hom_params['Lc_bar'] * eig_vec.conj().T @ F_matrix @ eig_vec )
    return coeff




omega0 = np.linspace(0.01, 0.99/( 1 + theta_tilde**2)**0.5*0.99, 500) /ref_scales['t0']
focusing = []
for om0 in omega0:
    v_g, P = P_vg_fun_dim('acoustic')
    focusing.append(P(om0) * Q_fun_dim(om0))
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(omega0, focusing)
ax[0].set_xlabel('Nondim Frequency $\Omega$')
ax[0].set_ylabel('Focusing/Defocusing Coefficient $P Q$')
peak_focus = omega0[np.argmax(focusing)]
print('Peak focusing at nondim frequency Omega = ', peak_focus)
print('Peak focusing at dim frequency f = ', peak_focus  / 2 / np.pi, ' Hz')
print("wave length at peak focusing: ", 2 * np.pi / q_fun_dim(peak_focus ) *1e3, ' mm')
# 1/( 1 + `theta_tilde**2)**0.5
ax[1].plot(q_fun_dim(omega0), omega0)
ax[1].scatter(q_fun_dim(peak_focus), peak_focus, color='red', label='Peak Focusing')
# q4_fun(0.5)
# q0 = q_fun_ndim(omega0)
# eigen_vector_ndim(omega0).conj().T @ F(-1j * omega0, 1j*q0) @ eigen_vector_ndim(omega0)
# eigen_vector_ndim(omega0).shape
# eigen_vector_ndim(omega0).conj() @ eigen_vector_ndim(omega0)
# F(-1j * omega0, 1j*q0) @ eigen_vector_ndim(omega0)
print('Q at peak focusing: ', Q_fun_dim(peak_focus))
print('P at peak focusing: ', P_vg_fun_dim('acoustic')[1](peak_focus))

In [ ]:
def envelope(omega0, eps, phi0):
	vg, P = P_vg_fun_dim('acoustic')
	Q0 = Q_fun_dim(omega0)
	vg0 = vg(omega0)
	P0 = P(omega0)
	q0 = q_fun_dim(omega0)
	print('At omega0= ', omega0, ': vg0= ', vg0, ', P0= ', P0, ', Q0= ', Q0, ', q0= ', q0)
	def A_func(X, T):
		sech_arg = eps * phi0 * np.sqrt(Q0 / (2 * P0)) * (X - vg0 * T)
		phase_arg = q0 * X - omega0 * T
		decay = np.exp( 1/2 * eps**2 * Q0 * phi0**2 * T )
		A = eps * phi0 * 1/np.cosh(sech_arg) * np.exp(1j * phase_arg) * decay
		return A
	return A_func
print('omega0= ', 1/ref_scales['t0']/2/np.pi)
envelope_func = envelope(peak_focus, eps=0.5, phi0=1.0)

t_eval = np.linspace(0, 800 * ref_scales['t0'], 800*100)

for t in t_eval[::len(t_eval)//5]:
	fig = plt.figure(figsize=(8, 5))
	A_t = envelope_func(x_eval, t)
	plt.plot(x_eval, np.real(A_t), label=f't={t/ref_scales["t0"]:0.1f} t0')
	plt.xlabel('x [mm]')
	plt.ylabel('Re{A(x,t)}')
	plt.title(f'Envelope at time t {t/ref_scales["t0"]:0.1f} t0')
	plt.legend()
	# plt.ylim([-0.02, 0.02])
	plt.show()
	# clear_output(wait=True)


In [ ]:
print("length of t_eval: ", len(t_eval) )
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
envelope_func = envelope(peak_focus, eps=0.1, phi0=2.0)

def make_v_exc():
    flux_linkage = envelope_func(4,
                                  t_eval#+t_eval[-1]/2
                                  )
    voltage = cumulative_trapezoid(np.real(flux_linkage), t_eval, initial=0) 
    v = interp1d(t_eval, voltage, kind='cubic', fill_value='extrapolate')
    return v
v_exc = make_v_exc()



plt.figure()
plt.plot(t_eval, v_exc(t_eval), '-' )
plt.xlabel('Time [s]')
plt.ylabel('Excitation Voltage [V]')
plt.show()


In [ ]:





results = rom.run_time_sim(v_exc=v_exc, j_exc=299, K_c=K_c, K_p=K_p, K_i=K_i, t_end=t_eval[-1], x_eval=x_eval, t_eval=t_eval)
t = results['t']
veloc = results['veloc']
freq = results['freq']
Y = results['Y']
X = results['X']
FRF = results['FRF']

# freq_modal, vel_mag, disp_mag, veloc_frq = rom.frequency_response(j_exc=j_exc, K_c=K_c, K_p=K_p, K_i=K_i,
# 										w=np.linspace(f0, f1, 1000)*2*np.pi, x_eval=x_eval)
# =================== Velocity field at multiple time instances ===================
# Select a few time indices to visualize the spatial velocity profile evolution


n_times = len(t)
# Choose time instances: start, several intermediate times, and end
time_indices = np.linspace(0, n_times - 1, 20, dtype=int)
# time_indices = np.arange(0, 20)  # specific time indices
time_values = t[time_indices]

# Create subplots for each time instance
fig, axes = plt.subplots(len(time_indices), 1, figsize=(10, 3 * len(time_indices)))
if len(time_indices) == 1:
    axes = [axes]

for idx, (ax, t_idx) in enumerate(zip(axes, time_indices)):
    ax.plot(x_eval, veloc[:, t_idx], 'b-', linewidth=2)
    ax.set_ylabel('Velocity [m/s]')
    ax.set_title(f'Time t = {time_values[idx]:.3f} s')
    ax.grid(True)

axes[-1].set_xlabel('Position along beam [m]')
plt.tight_layout()
plt.show()

# Alternative: overlay all on same plot
plt.figure(figsize=(10, 4))
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))
for t_idx, color, t_val in zip(time_indices, colors, time_values):
    plt.plot(x_eval, veloc[:, t_idx], color=color, linewidth=2, label=f't = {t_val:.3f} s')
plt.xlabel('Position along beam [m]')
plt.ylabel('Velocity [m/s]')
plt.title('Velocity Profile Evolution')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
veloc.shape
# t_eval.shape

In [ ]:
import importlib
from Modeling.models import plotting
importlib.reload(plotting)

plotting.animate_field_1d_pyvista(
	t=t_eval[:1000],
	u=veloc.T[:1000, :],
	# x=fe.x_nodes_free,
	filename="./anim/beam_displacement3.mp4",
	scale=2000,
    stride=1,
	ylabel="Displacement [m]",
	title="Beam transverse displacement"
)


In [ ]:
import subprocess
subprocess.run(["ffmpeg", "-version"], check=True)
